In [3]:
import pandas as pd

# 1) 데이터 불러오기
df = pd.read_excel("final_list(net and box_office).xlsx")

# 2) 날짜형으로 변환
df["개봉일"] = pd.to_datetime(df["개봉일"], errors="coerce")
df["넷플릭스 공개일"] = pd.to_datetime(df["넷플릭스 공개일"], errors="coerce")

# 3) 넷플릭스 공개일 - 개봉일  ⇒ 일(day) 단위 차이 계산
df["release_gap_days"] = (df["넷플릭스 공개일"] - df["개봉일"]).dt.days

# (선택) 넷플릭스가 극장보다 먼저 공개된 경우 음수가 나오는데,
# 필요하면 절댓값(양수)로 바꾸려면 아래 한 줄만 추가
# df["release_gap_days"] = df["release_gap_days"].abs()

# 확인
print(df[["개봉일", "넷플릭스 공개일", "release_gap_days"]].head())

         개봉일   넷플릭스 공개일  release_gap_days
0 2020-01-22 2020-04-21                90
1 2020-08-05 2023-01-10               888
2 2020-01-22 2020-08-19               210
3 2020-08-26 2024-09-14              1480
4 2020-06-24 2020-09-08                76


In [15]:
df

,순위,영화명,영화명(영문),개봉일,Netflix ID,넷플릭스 공개일,대표국적,국적(contries),매출액,매출액 점유율,관객수,스크린수,상영횟수,배급사,유형,장르,감독(director),박스오피스 기준년도,Unnamed: 18,release_gap_days
0,1,남산의 부장들,The Man Standing Next,2020-01-22,81259473,2020-04-21,한국,한국,41225216650,0.081,4750345,1659,140051,(주)쇼박스,장편,드라마,우민호,2020,NaN,90
1,2,다만 악에서 구하소서,DELIVER US FROM EVIL,2020-08-05,81370441,2023-01-10,한국,한국,38602260990,0.076,4357803,1998,193842,(주)씨제이이엔엠,장편,"범죄,액션",홍원찬,2020,NaN,888
2,4,히트맨,HITMAN: AGENT JUN,2020-01-22,81313503,2020-08-19,한국,한국,20614278000,0.040,2406232,1122,87782,롯데컬처웍스(주)롯데엔터테인먼트,장편,"코미디,액션",최원섭,2020,NaN,210
3,5,테넷,Tenet,2020-08-26,81198930,2024-09-14,미국,미국,18396929850,0.036,1992214,2228,164758,워너브러더스 코리아(주),장편,"액션,SF",크리스토퍼 놀란,2020,NaN,1480
4,7,#살아있다,#ALIVE,2020-06-24,81240831,2020-09-08,한국,한국,15968219900,0.031,1903992,1882,137073,롯데컬처웍스(주)롯데엔터테인먼트,장편,드라마,조일형,2020,NaN,76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,685,싱글 인 서울,Single in Seoul,2023-11-29,81629365,2024-02-27,한국,한국,10023984,0.000,1106,9,85,롯데컬처웍스(주)롯데엔터테인먼트,장편,"멜로/로맨스,코미디",박범수,2024,NaN,90
230,5,승부,The Match,2025-03-26,81292345,2025-05-08,한국,한국,19981575460,0.071,2138814,1773,148691,(주)바이포엠스튜디오,장편,드라마,김형주,2025,NaN,43
231,95,더 퍼스트 슬램덩크,The First Slam Dunk,2023-01-04,81765372,2025-03-02,일본,일본,233796260,0.001,18617,91,786,(주)넥스트엔터테인먼트월드(NEW),장편,애니메이션,이노우에 다케히코,2025,NaN,788
232,124,에브리씽 에브리웨어 올 앳 원스,Everything Everywhere All At Once,2022-10-12,81569721,2025-03-01,미국,미국,111942500,0.000,10379,27,233,워터홀컴퍼니(주),장편,"액션,코미디","다니엘 콴,다니엘 쉐이너트",2025,NaN,871


In [17]:
# 4) 평균 간격 요약 ─────────────────────────────────────────────────────
mean_all = df["release_gap_days"].mean()                              # 전체 평균
mean_pos = df.loc[df["release_gap_days"] >= 0, "release_gap_days"].mean()  # 극장→넷플릭스만

print(f"▸ 평균 간격(전체)        : {mean_all:,.1f} day")
print(f"▸ 평균 간격(극장→넷플릭스) : {mean_pos:,.1f} day")

▸ 평균 간격(전체)        : 348.1 day
▸ 평균 간격(극장→넷플릭스) : 348.1 day


In [19]:
# ── 선공개(음수 간격) 삭제 후 348일 미만 작품 수 ─────────────────────
n_lt_348 = df.query("release_gap_days < 348").shape[0]   # → 150
print(f"★ 348일 미만 간격 작품 : {n_lt_348:,}편")        # ★ 348일 미만 간격 작품 : 150편

★ 348일 미만 간격 작품 : 150편


# 영화 url 저장 코드

In [8]:
import pandas as pd
from urllib.parse import quote_plus   # URL 인코딩용

# 0) 데이터 로드
df = pd.read_excel("final_list(net and box_office).xlsx")

# 1) ‘영화명’ → 네이버 관람평 검색 URL 생성
base = "https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query="
df["naver_search_url"] = df["영화명"].apply(
    lambda title: f"{base}{quote_plus(f'영화 {title} 관람평')}"
)

# 2) (선택) 결과 확인 & 저장
print(df[["영화명", "naver_search_url"]].head())       # 앞부분 미리보기

df.to_excel("final_with_url.xlsx", index=False)      # 새 파일로 저장


           영화명                                   naver_search_url
0      남산의 부장들  https://search.naver.com/search.naver?where=ne...
1  다만 악에서 구하소서  https://search.naver.com/search.naver?where=ne...
2          히트맨  https://search.naver.com/search.naver?where=ne...
3           테넷  https://search.naver.com/search.naver?where=ne...
4        #살아있다  https://search.naver.com/search.naver?where=ne...


# 실관람객 크롤링